# Step 4 — Hyperparameter Tuning
**Course:** AI – Machine Learning Foundations  
**Professor:** Matteo Turilli  
**Date:** May 2026

This section performs hyperparameter tuning on the best-performing baseline models (Random Forest and Gradient Boosting) identified in Step 3 (Haya). Tuning is done using RandomizedSearchCV with 5-fold stratified cross-validation, applied only on the training set. The test set is never used during tuning.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score
from sklearn.metrics import balanced_accuracy_score

## Data Loading
We load the cleaned amphibian dataset prepared by the team. The target variable is `High_Risk`, where 1 = high extinction risk and 0 = low extinction risk.

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_excel('amphibian_cleaned.xlsx')

print("Dataset shape:", df.shape)
print("\nTarget distribution:")
print(df['High_Risk'].value_counts())
print("\nTarget proportions:")
print(df['High_Risk'].value_counts(normalize=True).round(3))

Saving amphibian_cleaned.xlsx to amphibian_cleaned.xlsx
Dataset shape: (7063, 32)

Target distribution:
High_Risk
0    4190
1    2873
Name: count, dtype: int64

Target proportions:
High_Risk
0    0.593
1    0.407
Name: proportion, dtype: float64


## Train-Test Split
The dataset is split into 80% training and 20% test before any tuning. The test set is locked away and will only be used once at the very end for final evaluation. Stratification ensures the class ratio is preserved in both sets.

In [ ]:
X = df.drop(columns=['High_Risk', 'Species_Name'])
y = df['High_Risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")
print(f"\nTrain class ratio: {y_train.mean().round(3)}")
print(f"Test class ratio:  {y_test.mean().round(3)}")

Training set: (5650, 30)
Test set:     (1413, 30)

Train class ratio: 0.407
Test class ratio:  0.407


## Feature Sets
We use the same two feature sets defined by Karim in Step 1:
- **Restricted:** taxonomy + geography + biology only (clean early-indicator benchmark, no leakage risk)
- **Full:** restricted + threat variables (stronger signal but potential leakage from IUCN assessment process)

Tuning is run on both so Noor can compare them in the final evaluation.

In [ ]:
taxonomy_features = ['Order', 'Family']

geography_features = [
    'Afrotropical', 'Australasian/Oceanian', 'Indomalayan',
    'Nearctic', 'Neotropical', 'Palearctic'
]

biology_features = [
    'Egg_Laying', 'Free_Living_Larval_Stage',
    'Live_Birth', 'Water_Breeding'
]

threat_features = [
    'Agriculture', 'Timber_and_plant_harvesting', 'Infrastructure_development',
    'Pollution', 'Mining/energy_production', 'Water_management',
    'Human_disturbance', 'Geological_Events', 'Over-exploitation',
    'Climate_(ongoing)', 'Climate_(future)', 'Fire',
    'Bd_(future)', 'Bd_(ongoing)', 'Bsal_(future)', 'Bsal_(ongoing)',
    'Invasive_species', 'Natives_species'
]

restricted_features = taxonomy_features + geography_features + biology_features
full_features = restricted_features + threat_features

print(f"Restricted feature count: {len(restricted_features)}")
print(f"Full feature count:       {len(full_features)}")

Restricted feature count: 12
Full feature count:       30


## Preprocessing Pipeline
Categorical variables (Order, Family) are one-hot encoded. Binary/numeric variables are imputed with 0. All preprocessing is fitted only on the training data inside the pipeline to prevent leakage.

In [ ]:
def build_preprocessor(feature_list):
    cat_cols = [c for c in feature_list if c in ['Order', 'Family']]
    num_cols = [c for c in feature_list if c not in cat_cols]

    transformers = []

    if cat_cols:
        transformers.append(('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols))

    if num_cols:
        transformers.append(('num', SimpleImputer(strategy='constant', fill_value=0), num_cols))

    return ColumnTransformer(transformers=transformers, remainder='drop')
print("Preprocessor ready ")

Preprocessor ready 


## Hyperparameter Tuning
We tune Random Forest and Gradient Boosting using RandomizedSearchCV with 5-fold stratified cross-validation. Tuning is performed entirely on the training set. The scoring metric is F1, chosen because the dataset has class imbalance and accuracy alone would be misleading.

We use RandomizedSearchCV over GridSearchCV because it covers a broader parameter space efficiently within a reasonable time budget, which is appropriate given our dataset size.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_param_grid = {
    'model__n_estimators':      [100, 200, 300],
    'model__max_depth':         [None, 10, 20],
    'model__min_samples_leaf':  [1, 2, 5],
    'model__class_weight':      ['balanced']
}

gbm_param_grid = {
    'model__n_estimators':   [100, 200],
    'model__learning_rate':  [0.05, 0.1, 0.2],
    'model__max_depth':      [3, 5],
    'model__subsample':      [0.8, 1.0],
}


## Tuning Random Forest — Restricted Feature Set
This is the clean benchmark model. It uses only taxonomy, geography, and biology features with no leakage risk.

In [ ]:
print("Tuning Random Forest on Restricted features")
print("This may take 2-3 minutes")

rf_pipe_restricted = Pipeline([
    ('preprocessor', build_preprocessor(restricted_features)),
    ('model', RandomForestClassifier(random_state=42))
])

rf_search_restricted = RandomizedSearchCV(
    rf_pipe_restricted,
    param_distributions=rf_param_grid,
    n_iter=15,
    scoring='f1',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search_restricted.fit(X_train[restricted_features], y_train)

print("\nDone!")
print(f"Best CV F1:    {rf_search_restricted.best_score_:.3f}")
print(f"Best params:   {rf_search_restricted.best_params_}")

Tuning Random Forest on Restricted features
This may take 2-3 minutes
Fitting 5 folds for each of 15 candidates, totalling 75 fits

 Done!
Best CV F1:    0.638
Best params:   {'model__n_estimators': 200, 'model__min_samples_leaf': 1, 'model__max_depth': None, 'model__class_weight': 'balanced'}


## Tuning Gradient Boosting — Restricted Feature Set
Gradient Boosting is tuned on the same restricted feature set for comparison. Unlike Random Forest, GBM does not support class_weight directly — instead we control overfitting through learning rate and subsampling.

In [ ]:
print("Tuning Gradient Boosting on Restricted features")
print("This may take 2-3 minutes")

gbm_pipe_restricted = Pipeline([
    ('preprocessor', build_preprocessor(restricted_features)),
    ('model', GradientBoostingClassifier(random_state=42))
])

gbm_search_restricted = RandomizedSearchCV(
    gbm_pipe_restricted,
    param_distributions=gbm_param_grid,
    n_iter=15,
    scoring='f1',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

gbm_search_restricted.fit(X_train[restricted_features], y_train)

print("\n Done!")
print(f"Best CV F1:    {gbm_search_restricted.best_score_:.3f}")
print(f"Best params:   {gbm_search_restricted.best_params_}")

Tuning Gradient Boosting on Restricted features
This may take 2-3 minutes
Fitting 5 folds for each of 15 candidates, totalling 75 fits

 Done!
Best CV F1:    0.614
Best params:   {'model__subsample': 0.8, 'model__n_estimators': 200, 'model__max_depth': 3, 'model__learning_rate': 0.2}


## Tuning Random Forest — Full Feature Set
The full feature set includes threat variables. As noted in the leakage analysis, these variables may be co-determined with the IUCN assessment process, so results here are expected to be inflated. We include them for comparison purposes only.

In [ ]:
print("Tuning Random Forest on Full features")
print("This may take 2-3 minutes")

rf_pipe_full = Pipeline([
    ('preprocessor', build_preprocessor(full_features)),
    ('model', RandomForestClassifier(random_state=42))
])

rf_search_full = RandomizedSearchCV(
    rf_pipe_full,
    param_distributions=rf_param_grid,
    n_iter=15,
    scoring='f1',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search_full.fit(X_train[full_features], y_train)

print("\nDone!")
print(f"Best CV F1:    {rf_search_full.best_score_:.3f}")
print(f"Best params:   {rf_search_full.best_params_}")

Tuning Random Forest on Full features
This may take 2-3 minutes
Fitting 5 folds for each of 15 candidates, totalling 75 fits

Done!
Best CV F1:    0.997
Best params:   {'model__n_estimators': 100, 'model__min_samples_leaf': 2, 'model__max_depth': 20, 'model__class_weight': 'balanced'}


## Tuning Gradient Boosting — Full Feature Set
Final tuning run. Again, near-perfect scores on the full feature set are expected due to the leakage concern identified earlier.

In [ ]:
print("Tuning Gradient Boosting on Full features")
print("This may take 2-3 minutes")

gbm_pipe_full = Pipeline([
    ('preprocessor', build_preprocessor(full_features)),
    ('model', GradientBoostingClassifier(random_state=42))
])

gbm_search_full = RandomizedSearchCV(
    gbm_pipe_full,
    param_distributions=gbm_param_grid,
    n_iter=15,
    scoring='f1',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

gbm_search_full.fit(X_train[full_features], y_train)

print("\nDone!")
print(f"Best CV F1:    {gbm_search_full.best_score_:.3f}")
print(f"Best params:   {gbm_search_full.best_params_}")

Tuning Gradient Boosting on Full features
This may take 2-3 minutes
Fitting 5 folds for each of 15 candidates, totalling 75 fits

Done!
Best CV F1:    0.994
Best params:   {'model__subsample': 0.8, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.2}


## Results Summary
We evaluate each tuned model on the held-out test set. The best model is selected based on F1 score on the Restricted feature set, as this represents the honest benchmark free from leakage risk.

In [ ]:
def evaluate(search, X_test, y_test, feature_list):
    best = search.best_estimator_
    y_pred = best.predict(X_test[feature_list])
    y_prob = best.predict_proba(X_test[feature_list])[:, 1]
    return {
        'Best CV F1':       round(search.best_score_, 3),
        'Test F1':          round(f1_score(y_test, y_pred), 3),
        'Recall':           round(recall_score(y_test, y_pred), 3),
        'Precision':        round(precision_score(y_test, y_pred), 3),
        'ROC-AUC':          round(roc_auc_score(y_test, y_prob), 3),
        'Balanced Acc':     round(balanced_accuracy_score(y_test, y_pred), 3),
    }

results = [
    {'Feature Set': 'Restricted', 'Model': 'Random Forest',     **evaluate(rf_search_restricted,  X_test, y_test, restricted_features)},
    {'Feature Set': 'Restricted', 'Model': 'Gradient Boosting', **evaluate(gbm_search_restricted, X_test, y_test, restricted_features)},
    {'Feature Set': 'Full',       'Model': 'Random Forest',     **evaluate(rf_search_full,         X_test, y_test, full_features)},
    {'Feature Set': 'Full',       'Model': 'Gradient Boosting', **evaluate(gbm_search_full,        X_test, y_test, full_features)},
]

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

Feature Set             Model  Best CV F1  Test F1  Recall  Precision  ROC-AUC  Balanced Acc
 Restricted     Random Forest       0.638    0.618   0.675      0.571    0.728         0.663
 Restricted Gradient Boosting       0.614    0.581   0.565      0.597    0.727         0.652
       Full     Random Forest       0.997    0.992   0.984      1.000    0.996         0.992
       Full Gradient Boosting       0.994    0.991   0.983      1.000    0.996         0.991


## Comparison with Baseline (Haya's Results)
To measure the impact of tuning, we compare our tuned results against Haya's untuned baseline models on the same feature sets.

In [ ]:
baseline = {
    'Feature Set': ['Restricted', 'Restricted', 'Full', 'Full'],
    'Model': ['Random Forest', 'Gradient Boosting', 'Random Forest', 'Gradient Boosting'],
    'Baseline F1': [0.619, 0.565, 0.992, 0.988],
    'Tuned F1':    [0.618, 0.581, 0.992, 0.991],
}

comparison_df = pd.DataFrame(baseline)
comparison_df['Improvement'] = (comparison_df['Tuned F1'] - comparison_df['Baseline F1']).round(3)
print(comparison_df.to_string(index=False))

Feature Set             Model  Baseline F1  Tuned F1  Improvement
 Restricted     Random Forest        0.619     0.618       -0.001
 Restricted Gradient Boosting        0.565     0.581        0.016
       Full     Random Forest        0.992     0.992        0.000
       Full Gradient Boosting        0.988     0.991        0.003


## Best Hyperparameters Discussion

**Random Forest (Restricted):** `max_depth=None` means trees are fully grown, allowing the model to capture complex patterns in the data. `n_estimators=200` provides enough trees for stable predictions. `class_weight=balanced` compensates for the 59/41 class imbalance.

**Gradient Boosting (Restricted):** `learning_rate=0.2` with `max_depth=3` and `subsample=0.8` is a classic combination — shallow trees with moderate learning rate reduces overfitting while subsampling adds randomness for better generalisation.

**Full feature set:** Near-perfect scores (0.99+) on both models confirm the leakage concern. These results are reported for completeness but the Restricted model is the reliable benchmark.

## Tuning Observation
The improvement from tuning is modest on the Restricted feature set. Random Forest showed no meaningful gain (-0.001), suggesting the default settings were already near-optimal for this feature space. Gradient Boosting improved slightly (+0.016).

This is a valid and honest finding. On a structured binary dataset with 12 features, tree-based models can reach near-optimal performance without extensive tuning. The real value of this step is confirming that the baseline results were reliable and not a product of lucky default settings.

## Best Model Selection
On the Restricted feature set, tuned Random Forest achieves the highest F1 (0.618) and ROC-AUC (0.728), making it the selected model for final evaluation. The Full feature set results are noted but treated with caution due to likely leakage from the IUCN assessment process.

In [ ]:
import joblib

results_df.to_csv('hamza_tuning_results.csv', index=False)
print("Results saved as hamza_tuning_results.csv")

joblib.dump(rf_search_restricted.best_estimator_,  'best_rf_restricted.pkl')
joblib.dump(gbm_search_restricted.best_estimator_, 'best_gbm_restricted.pkl')
joblib.dump(rf_search_full.best_estimator_,        'best_rf_full.pkl')
joblib.dump(gbm_search_full.best_estimator_,       'best_gbm_full.pkl')
print("All 4 tuned pipelines saved")

Results saved as hamza_tuning_results.csv
All 4 tuned pipelines saved


In [ ]:
from google.colab import files
files.download('hamza_tuning_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download('best_rf_restricted.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download('best_rf_full.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download('best_gbm_full.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Generate test data files for Noor
X_test[restricted_features].to_csv('X_test_restricted.csv', index=False)
X_test[full_features].to_csv('X_test_full.csv', index=False)
y_test.to_csv('y_test.csv', index=False)
print("Test data files ready")

✅ Test data files ready


In [ ]:
files.download('X_test_restricted.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download('X_test_full.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download('y_test.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import shutil
import joblib
from google.colab import files

# Create folder
os.makedirs('Hamza_Final_Outputs_For_Noor', exist_ok=True)

# Save all files into the folder
results_df.to_csv('Hamza_Final_Outputs_For_Noor/hamza_tuning_results.csv', index=False)
X_test[restricted_features].to_csv('Hamza_Final_Outputs_For_Noor/X_test_restricted.csv', index=False)
X_test[full_features].to_csv('Hamza_Final_Outputs_For_Noor/X_test_full.csv', index=False)
y_test.to_csv('Hamza_Final_Outputs_For_Noor/y_test.csv', index=False)
joblib.dump(rf_search_restricted.best_estimator_,  'Hamza_Final_Outputs_For_Noor/best_rf_restricted.pkl')
joblib.dump(gbm_search_restricted.best_estimator_, 'Hamza_Final_Outputs_For_Noor/best_gbm_restricted.pkl')
joblib.dump(rf_search_full.best_estimator_,        'Hamza_Final_Outputs_For_Noor/best_rf_full.pkl')
joblib.dump(gbm_search_full.best_estimator_,       'Hamza_Final_Outputs_For_Noor/best_gbm_full.pkl')
print("All 8 files saved to folder")

# Zip and download
shutil.make_archive('Hamza_Final_Outputs_For_Noor', 'zip', 'Hamza_Final_Outputs_For_Noor')
files.download('Hamza_Final_Outputs_For_Noor.zip')
print("Folder downloaded as zip")

✅ All 8 files saved to folder


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Folder downloaded as zip
